# Experiment To Validate Normalization Tokenizer Choice

## Imports

In [51]:
from pathlib import Path

import pandas as pd

from src.asr.normalize import (
    get_tokenizer,
    normalize_text,
)

from src.asr.metrics import compute_wer

## Load Experiment Manifest and Transcriprion Results

In [52]:
RUN_DIR = Path(
    "../results/asr/runs/study_base_long"
)

manifest = pd.read_csv(
    RUN_DIR / "manifest_snapshot.csv"
)

transcriptions = pd.read_csv(
    RUN_DIR / "transcriptions.csv"
)

# Only uses successful transcriptions
transcriptions = transcriptions.loc[
    transcriptions["status"] == "ok"
].copy()

### Combine Results
Each row contains:
* sample_id
* dataset
* reference_transcript
* transcription_raw

In [53]:
data = manifest[
    [
        "sample_id",
        "dataset",
        "reference_transcript",
    ]
].merge(
    transcriptions[
        [
            "sample_id",
            "transcription_raw",
        ]
    ],
    on="sample_id",
    how="inner",
    validate="one_to_one",
)

## Define Tokenizer Models for the Experiment

In [54]:
NORMALIZER_MODELS = {
    "tiny": "openai/whisper-tiny",
    "base": "openai/whisper-base",
    "small": "openai/whisper-small",
    "medium": "openai/whisper-medium",
}

## Normalize Each Reference and Transcript Using Each Tokenizer

In [55]:
for name, model_name in NORMALIZER_MODELS.items():

    print(f"Testing {model_name}...")

    tokenizer = get_tokenizer(
        model_name
    )

    data[f"reference_norm_{name}"] = (
        data.apply(
            lambda row:
                normalize_text(
                    row["reference_transcript"],
                    row["dataset"],
                    tokenizer=tokenizer,
                )[0],
            axis=1,
        )
    )

    data[f"hypothesis_norm_{name}"] = (
        data.apply(
            lambda row:
                normalize_text(
                    row["transcription_raw"],
                    row["dataset"],
                    tokenizer=tokenizer,
                )[0],
            axis=1,
        )
    )

Testing openai/whisper-tiny...
Testing openai/whisper-base...
Testing openai/whisper-small...
Testing openai/whisper-medium...


# Comparison 
Using whisper-small as the baseline

In [56]:
BASELINE = "small"

In [57]:
comparison_results = []

for name in NORMALIZER_MODELS:

    if name == BASELINE:
        continue

    reference_different = (
        data[f"reference_norm_{name}"]
        != data[
            f"reference_norm_{BASELINE}"
        ]
    )

    hypothesis_different = (
        data[f"hypothesis_norm_{name}"]
        != data[
            f"hypothesis_norm_{BASELINE}"
        ]
    )

    comparison_results.append(
        {
            "normalizer": name,
            "n_samples": len(data),

            "reference_differences":
                reference_different.sum(),

            "hypothesis_differences":
                hypothesis_different.sum(),
        }
    )

In [58]:
comparison_summary = pd.DataFrame(
    comparison_results
)

comparison_summary

,normalizer,n_samples,reference_differences,hypothesis_differences
0,tiny,409,0,0
1,base,409,0,0
2,medium,409,0,0


### Differences

In [59]:
different = data.loc[
    (
        data["reference_norm_tiny"]
        != data["reference_norm_small"]
    )
    |
    (
        data["hypothesis_norm_tiny"]
        != data["hypothesis_norm_small"]
    )
]

different[
    [
        "sample_id",
        "dataset",
        "reference_transcript",
        "reference_norm_tiny",
        "reference_norm_small",
        "transcription_raw",
        "hypothesis_norm_tiny",
        "hypothesis_norm_small",
    ]
]

if different.empty:
    print("No differences between models")
else:
    print(f"Found {len(different)} differences:")
    display(different)

No differences between models


### WER Comparison

In [60]:
for name in NORMALIZER_MODELS:

    data[f"wer_{name}"] = data.apply(
        lambda row:
            compute_wer(
                row[f"reference_norm_{name}"],
                row[f"hypothesis_norm_{name}"],
            ).wer,
        axis=1,
    )

In [61]:
for name in NORMALIZER_MODELS:

    n_different = (
        data[f"wer_{name}"]
        != data["wer_small"]
    ).sum()

    print(
        f"{name}: "
        f"{n_different} WER differences"
    )

tiny: 0 WER differences
base: 0 WER differences
small: 0 WER differences
medium: 0 WER differences


## Test Assertion 
Checks every normalizer gets the exact same result for every tested sample.

In [62]:
for name in NORMALIZER_MODELS:

    assert (
        data[f"reference_norm_{name}"]
        == data["reference_norm_small"]
    ).all()

    assert (
        data[f"hypothesis_norm_{name}"]
        == data["hypothesis_norm_small"]
    ).all()

## Conclusion
* Normalization was evaluated using the Whisper tiny, base, small, and medium tokenizers across both the reference transcripts and ASR hypotheses in the SAA study dataset. 
* All tested tokenizers produced identical normalized text and consequently identical WER values; therefore, normalization is independent of the tested Whisper checkpoint size for the study data.
* A single fixed Whisper tokenizer was therefore used for all subsequent scoring. 
* The tokenizer revision and the Transformers package version are pinned and recorded with each scoring run to ensure reproducibility.

In [63]:
from huggingface_hub import HfApi

api = HfApi()

info = api.model_info(
    "openai/whisper-small"
)

print(info.sha)

973afd24965f72e36ca33b3055d56a652f456b4d
